# 🧪 POC — LLM Local no Google Colab
## Rodando Qwen3-4B sem API externa, direto no Colab!

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/poc_llm_local_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivo:** Provar que é possível rodar um modelo pequeno (4B parâmetros)
direto no Google Colab gratuito (T4 GPU), sem precisar de API key externa.

**Status:** POC experimental — não é material de aula (ainda).

---


## 1. Verificar GPU

⚠️ **Importante:** Vá em `Runtime → Change runtime type` e selecione **T4 GPU** antes de continuar.


In [ ]:
# Verificar GPU disponível
!nvidia-smi


## 2. Instalar llama-cpp-python (com suporte CUDA)

O `llama-cpp-python` é um binding Python do `llama.cpp` — o motor de inferência mais leve e rápido para modelos locais.

Vamos instalar com suporte a GPU (CUDA) pra aproveitar a T4.


In [ ]:
# Instalar llama-cpp-python com suporte CUDA
# (pode demorar 1-2 min compilando)

!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --force-reinstall --no-cache-dir

print("✅ llama-cpp-python instalado com CUDA!")


## 3. Baixar o Modelo GGUF

Vamos usar o **Qwen3-4B** quantizado em Q4_K_M (~2.4 GB). Formato GGUF, otimizado para `llama.cpp`.

O download usa o `huggingface_hub` e leva ~1-2 min dependendo da velocidade do Colab.


In [ ]:
# Baixar modelo GGUF do Hugging Face
!pip install huggingface_hub

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Qwen/Qwen3-4B-GGUF",
    filename="qwen3-4b-q4_k_m.gguf",
    local_dir="/content/model"
)

print(f"✅ Modelo baixado: {model_path}")


## 4. Carregar o Modelo na GPU

Agora carregamos o modelo na memória. O parâmetro `n_gpu_layers=-1` manda todas as camadas pra GPU.


In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,       # todas as camadas na GPU
    n_ctx=4096,            # contexto (tokens)
    verbose=False
)

print("✅ Modelo carregado na GPU!")
print(f"    GPU layers: {llm.n_gpu_layers}")


## 5. Inferência — Geração Simples

Vamos testar a geração de texto. O modelo Qwen3 suporta "thinking" (raciocínio interno).

- **Com thinking:** o modelo mostra o raciocínio antes de responder (mais lento, mais preciso)
- **Sem thinking:** resposta direta (mais rápido)


In [ ]:
# Geração simples (sem thinking - mais rápido)

response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
    ],
    max_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    extra_body={"enable_thinking": False}  # desativa thinking
)

resposta = response["choices"][0]["message"]["content"]
tokens_used = response["usage"]

print(f"🤖 Resposta: {resposta}")
print(f"📊 Tokens: prompt={tokens_used['prompt_tokens']}, completion={tokens_used['completion_tokens']}")


In [ ]:
# Geração COM thinking (raciocínio interno)

response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Quantos números primos existem entre 1 e 20?"}
    ],
    max_tokens=512,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    extra_body={"enable_thinking": True}  # ativa thinking
)

msg = response["choices"][0]["message"

# Extrair thinking e resposta
thinking = msg.get("reasoning_content", "(sem thinking)")
content = msg["content"]

print(f"💭 Thinking:\n{thinking}")
print(f"\n🤖 Resposta: {content}")


## 6. Benchmark Rápido

Vamos medir a velocidade de inferência no Colab gratuito.


In [ ]:
import time

# Benchmark: gerar 100 tokens e medir tempo
prompt = "Conte uma história curta sobre um robô que aprende a cozinhar."

start = time.time()
response = llm.create_chat_completion(
    messages=[{"role": "user", "content": prompt}],
    max_tokens=100,
    temperature=0.7,
    extra_body={"enable_thinking": False}
)
elapsed = time.time() - start

completion_tokens = response["usage"]["completion_tokens"]
speed = completion_tokens / elapsed

print(f"⏱️ Tempo: {elapsed:.1f}s")
print(f"📊 Tokens gerados: {completion_tokens}")
print(f"🚀 Velocidade: {speed:.1f} tokens/s")
print(f"\n📝 Resposta: {response['choices'][0]['message']['content']}")


## 7. Streaming — Resposta em Tempo Real

Streaming mostra o texto conforme é gerado, ao invés de esperar a resposta completa.


In [ ]:
# Streaming de resposta
print("🤖 ", end="")

for chunk in llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Você é um assistente útil. Responda em português."},
        {"role": "user", "content": "Explique o que é uma API em 3 frases."}
    ],
    max_tokens=128,
    temperature=0.7,
    stream=True,
    extra_body={"enable_thinking": False}
):
    delta = chunk["choices"][0]["delta"]
    if "content" in delta:
        print(delta["content"], end="", flush=True)

print()  # newline no final


## 8. 📋 Resultados da POC

### O que funciona ✅
- **Qwen3-4B roda no Colab gratuito** (T4 GPU, 16 GB VRAM)
- Modelo Q4_K_M ocupa ~2.4 GB de VRAM — sobra bastante espaço
- `llama-cpp-python` com CUDA compilou sem problemas
- Velocidade aceitável para demonstração (~15-25 tokens/s dependendo do contexto)
- Chat com memória funciona perfeitamente (mesmo padrão do LangChain)
- Streaming funciona e melhora a experiência

### Limitações ⚠️
- **Setup demorado:** ~3-5 min de instalação + download do modelo cada vez que o Colab reinicia
- **Sem persistência:** modelo precisa ser baixado a cada sessão
- **Rate limits do Colab:** sessões gratuitas têm limite de horas
- **Contexto limitado:** 4K tokens é pouco para RAG mais suficiente para chat básico
- **Velocidade:** não é tão rápida quanto API do Gemini (mas é local!)

### Comparação com Gemini API

| Aspecto | Gemini API | Qwen3-4B Local |
|---------|-----------|----------------|
| Setup | API key só | ~5 min instalação |
| Velocidade | Rápido (~100+ tok/s) | Médio (~15-25 tok/s) |
| Custos | Grátis com limites | 100% grátis |
| Privacidade | Dados vão pro Google | Fica no Colab |
| Qualidade | Muito alta | Boa para 4B |
| Offline | Não | Sim (no Colab) |

### Vabilidade para a Guilda
- **Para Aula 3 (amanhã):** Não usar ainda — muito setup pra primeira vez com Colab
- **Para Aula futura:** Viável como demonstração de "IA roda na sua máquina"
- **Recomendação:** Usar Gemini API como padrão, POC local como demonstração avulsa

---
*POC experimental — Guilda de IA UFVJM 2026.1*
